### **Sample Cash Flow Simulation**
The purpose of this notebook is to use developed models to create data-driven, Monte-Carlo style simulations of loan pool cash flows. This simulation uses results from SARIMAX to predict interest rates. Due to time constraints, the simulation uses a Random Forest to predict *both* prepayment and default, although the developed Random Forest is a worse predictor of defaults.

This code was created with error correction and writing assistance by Perplexity AI.

### **Step 1: Generate the loan pool**

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
#Read in SARIMAX monthly interest rates
rate_map = pd.read_csv("monthly_forecast_rates.csv")
# Filter predictions to 2022 months only
rate_2022 = rate_map[rate_map['Month'].str.contains('2022')]
rate_map_2022 = dict(zip(rate_2022['Month'], rate_2022['Predicted_Avg_Rate_%']))

# Generate 500 loans between $1000-$35000 using 2022 rates
np.random.seed(42)
df = pd.DataFrame({
    'idx': range(1, 501),
    'ListingCreationDate': np.random.choice(rate_2022['Month'], 500, replace=True),
    'LoanOriginalAmount': np.random.uniform(1000, 35000, 500).round(0).astype(int)
})

df['BorrowerRate'] = df['ListingCreationDate'].map(rate_map_2022)
#consistency scale
df['BorrowerRate'] = df['BorrowerRate']/100

print(df.head())


   idx ListingCreationDate  LoanOriginalAmount  BorrowerRate
0    1             2022-07               11588      0.124132
1    2             2022-04               34303      0.122143
2    3             2022-11                6961      0.128358
3    4             2022-08                1583      0.125079
4    5             2022-05               26954      0.122696


In [ ]:
# ===== FED PREP =====
fed_df = pd.read_csv("FEDFUNDS (1).csv")
fed_df["observation_date"] = pd.to_datetime(fed_df["observation_date"])
#Calculate fed rate at loan origin for each loan.
fed_monthly = fed_df.set_index('observation_date').resample('MS').interpolate('linear').ffill()
fed_dict = fed_monthly['FEDFUNDS'].to_dict()

In [ ]:
#Load in Random Forests
import joblib

# Load model
prepay_model = joblib.load('rf_prepay_model.pkl')
default_model = joblib.load('rf_default_model.pkl')
# Load features (assume list/array)
feature_names = joblib.load('rf_features.pkl')
print(feature_names)


['month_since_orig', 'apr_fed_spread', 'fed_level', 'borrower_apr', 'loan_amount_log']


In [ ]:
#Standard function for calculating loan income.
def pmt(rate, nper, pv):
    """Positive payment for lender inflow."""
    pv = abs(pv)
    if rate == 0:
        return pv / nper
    return rate * pv / (1 - (1 + rate) ** -nper)


df['TermMonths'] = 36
df['MonthlyRate'] = df['BorrowerRate'] / 12


def calculate_payment(principal, monthly_rate, term_months):
    if monthly_rate == 0:
        return principal / term_months
    return pmt(monthly_rate, term_months, principal)

df['ScheduledPayment'] = [calculate_payment(row['LoanOriginalAmount'], row['MonthlyRate'], row['TermMonths'])
                          for _, row in df.iterrows()]
df['CurrentBalance'] = df['LoanOriginalAmount'].copy()


## Step 2: Run simulation

In [ ]:
from datetime import datetime
start_date = pd.to_datetime('2022-01-01')
end_date = pd.to_datetime('2024-12-01')
months = pd.date_range(start=start_date, end=end_date, freq='MS')

# Setup
df['StartDate'] = pd.to_datetime(df['ListingCreationDate'] + '-01')
df['Originated'] = False
df['Dropped'] = False
df['CurrentBalance'] = df['LoanOriginalAmount'].copy()

# Simulation loop
np.random.seed(42)
cashflows = []

for month in months:
    monthly_cf = []

    # Counts: originated = started by month, active = originated & balance>0 & not dropped
    df.loc[(month >= df['StartDate']) & ~df['Originated'], 'Originated'] = True  # Mark new starts
    num_originated = df['Originated'].sum()
    active_mask = df['Originated'] & (df['CurrentBalance'] > 0) & ~df['Dropped']
    num_active = active_mask.sum()
    num_dropped = df['Dropped'].sum()

    if num_active == 0:
        break

    active_loans = df[active_mask].copy()

    for idx_row, loan in active_loans.iterrows():
        # Features set up
        orig_date = loan['StartDate']
        month_since_orig = (month.year - orig_date.year) * 12 + (month.month - orig_date.month)

        fed_rate = fed_monthly.loc[month, 'FEDFUNDS'] if month in fed_monthly.index else 0.5
        apr_fed_spread = loan['BorrowerRate'] - fed_rate

        features_df = pd.DataFrame([[month_since_orig, apr_fed_spread, fed_rate,
                                    loan['BorrowerRate'], np.log(loan['LoanOriginalAmount'])]],
                                   columns=feature_names)

        #random forest probabilities
        p_def = default_model.predict_proba(features_df)[0, 1]
        p_prepay = prepay_model.predict_proba(features_df)[0, 1]

        #Monte-Carlo style decision if loans prepay/default/continue based on RF probabilities
        u = np.random.uniform(0, 1)

        B = loan['CurrentBalance']
        r = loan['MonthlyRate']
        payment = loan['ScheduledPayment']  # Positive
        interest = B * r
        prin_scheduled = min(payment - interest, B)

        event = 'survive'
        cf = interest + prin_scheduled
        new_balance = B - prin_scheduled

        if u < p_def:
            event = 'default'
            cf = 0
            new_balance = 0
            df.loc[idx_row, 'Dropped'] = True
        elif u < p_def + p_prepay:
            event = 'prepay'
            cf = B
            new_balance = 0
            df.loc[idx_row, 'Dropped'] = True

        monthly_cf.append({
            'month': month,
            'loan_idx': loan['idx'],
            'beginning_balance': B,
            'p_default': p_def,
            'p_prepay': p_prepay,
            'u': u,
            'event': event,
            'interest': interest,
            'prin_scheduled': prin_scheduled,
            'prepay_prin': B if event == 'prepay' else 0,
            'total_cf': cf,
            'ending_balance': new_balance
        })

        df.loc[idx_row, 'CurrentBalance'] = new_balance

    monthly_df = pd.DataFrame(monthly_cf)

    # Scalars for totals
    total_interest = monthly_df['interest'].sum()
    total_prin_scheduled = monthly_df['prin_scheduled'].sum()
    total_prepay_prin = monthly_df['prepay_prin'].sum()
    total_cf = monthly_df['total_cf'].sum()

    monthly_total = pd.DataFrame({
        'month': [month],
        'total_cf': [total_cf],
        'interest': [total_interest],
        'prin_scheduled': [total_prin_scheduled],
        'prepay_prin': [total_prepay_prin],
        'num_active': [num_active],
        'num_originated': [num_originated],
        'num_dropped': [num_dropped]
    })

    cashflows.append(monthly_total)

    print(f"{month.date()}: Active {num_active}, Orig {num_originated}, Dropped {num_dropped}, CF ${total_cf:,.0f}")

cf_df = pd.concat(cashflows, ignore_index=True)
cf_df.to_csv('cashflows_with_counts.csv', index=False)
print(cf_df[['month', 'total_cf', 'num_active', 'num_originated', 'num_dropped']].head())


/tmp/ipykernel_2057/4292620654.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5244.629503490089' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[idx_row, 'CurrentBalance'] = new_balance


2022-01-01: Active 51, Orig 51, Dropped 0, CF $44,497
2022-02-01: Active 74, Orig 84, Dropped 10, CF $36,554
2022-03-01: Active 105, Orig 131, Dropped 26, CF $53,393
2022-04-01: Active 126, Orig 169, Dropped 43, CF $89,401
2022-05-01: Active 142, Orig 208, Dropped 66, CF $60,854
2022-06-01: Active 149, Orig 246, Dropped 97, CF $69,381
2022-07-01: Active 169, Orig 292, Dropped 123, CF $148,931
2022-08-01: Active 183, Orig 335, Dropped 152, CF $170,941
2022-09-01: Active 183, Orig 374, Dropped 191, CF $100,379
2022-10-01: Active 195, Orig 414, Dropped 219, CF $141,558
2022-11-01: Active 186, Orig 444, Dropped 258, CF $128,941
2022-12-01: Active 211, Orig 500, Dropped 289, CF $128,666
2023-01-01: Active 166, Orig 500, Dropped 334, CF $107,473
2023-02-01: Active 138, Orig 500, Dropped 362, CF $95,312
2023-03-01: Active 112, Orig 500, Dropped 388, CF $120,060
2023-04-01: Active 93, Orig 500, Dropped 407, CF $52,200
2023-05-01: Active 78, Orig 500, Dropped 422, CF $108,303
2023-06-01: Active

In [15]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(name='Cashflow ($k)', x=cf_df['month'], y=cf_df['total_cf']/1000, marker_color='#1f77b4'))
fig.add_trace(go.Scatter(name='Active', x=cf_df['month'], y=cf_df['num_active'], yaxis='y2', line=dict(color='#ff7f0e')))
fig.add_trace(go.Scatter(name='Dropped', x=cf_df['month'], y=cf_df['num_dropped'], yaxis='y2', line=dict(color='#d62728', dash='dash')))
fig.update_layout(
    title="Portfolio Cashflow & Status (MC Path)",
    xaxis_title="Month",
    yaxis_title="CF ($k)",
    yaxis2=dict(title="Loans", side='right', overlaying='y'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()